In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json
from core.ollama_client import chat_with_ollama
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents

MODEL = os.getenv("OLLAMA_MODEL")

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
  "links": [
    {"type": "about page", "url": "https://full.url/goes/here/about"},
    {"type": "careers page", "url": "https://another.full.url/careers"}
  ]
}
"""

In [ ]:
def get_links_user_prompt(url):
  user_prompt = f"""
  Here is the list of links on the website {url},
  Please decide which of these are relevant web links for a brochure about the company,
  respond with the full https URL in JSON format.

  """
  links = fetch_website_links(url)
  user_prompt += "\n".join(links)
  return user_prompt

In [ ]:
def select_relevant_links(url):
  messages = [
    {
        "role": "system",
        "content": link_system_prompt
    },
    {
        "role": "user",
        "content": get_links_user_prompt(url)
    }
  ]

  print(f"Selecting relevant links for {url} by calling {MODEL}")
  result = chat_with_ollama(
    messages=messages,
    timeout=300,
    response_format={"type": "json_object"}
  )

  parsed = json.loads(result)
  raw_links = parsed.get("links", []) if isinstance(parsed, dict) else []

  normalized_links = []
  for item in raw_links:
    if isinstance(item, dict):
      link_url = item.get("url")
      if link_url:
        normalized_links.append({
          "type": item.get("type", "relevant link"),
          "url": link_url,
        })
    elif isinstance(item, str):
      normalized_links.append({
        "type": "relevant link",
        "url": item,
      })

  links = {"links": normalized_links}
  print(f"Found {len(links['links'])} relevant links")
  return links

In [ ]:
# select_relevant_links("https://huggingface.co")

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        link_type = link.get("type", "relevant link")
        link_url = link.get("url")
        if not link_url:
            continue
        result += f"\n\n### Link: {link_type}\n"
        result += fetch_website_contents(link_url)
    return result

In [ ]:
# fetch_page_and_all_relevant_links("https://huggingface.co")

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
  user_prompt = f"""
  You are looking at a company called: {company_name}
  Here are the contents of its landing page and other relevant pages;
  use this information to build a short brochure of the company in markdown without code blocks.\n\n
  """
  user_prompt += fetch_page_and_all_relevant_links(url)
  return user_prompt[:10_000]

In [ ]:
# get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
def create_brochure(company_name, url):
  result = chat_with_ollama(
    messages=[
      {"role": "system", "content": brochure_system_prompt},
      {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ],
  )
  display(Markdown(result))

In [ ]:
# create_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
def stream_brochure(company_name, url):
  stream = chat_with_ollama(
    messages=[
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
      ],
    stream=True
  )
  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")